# Chapter 6 — Classification
**MADT6004 · Brew Lab BKK case**

Brew Lab sent the *Comeback 50* voucher to lapsed loyalty members. Some responded; most didn't. You'll train a classifier to predict who is likely to respond — useful for targeting future campaigns.

You will:
1. Build features per customer
2. Train a logistic regression and a decision tree
3. Compare predicted probabilities

(Detailed evaluation comes in Chapter 7a.)


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Build the feature table
Per customer, compute simple lifetime features and join the campaign response.

In [ ]:
feat = pd.read_sql("""
SELECT c.customer_id,
       c.acquisition_channel,
       COUNT(t.order_id) AS visits,
       COALESCE(SUM(t.total), 0) AS lifetime_spend,
       COALESCE(AVG(t.total), 0) AS avg_ticket
FROM customers c
LEFT JOIN transactions t ON t.customer_id = c.customer_id
GROUP BY c.customer_id
""", conn)
resp = pd.read_sql("SELECT customer_id, responded FROM campaign_responses", conn)
df = feat.merge(resp, on="customer_id", how="inner")  # only members targeted by the campaign
print("Targeted members:", len(df))
print("Response rate :", df["responded"].mean().round(3))
print(df.head())


## 3. Train / test split
Stratify on the target so both halves have the same response rate.

In [ ]:
X = pd.get_dummies(df[["acquisition_channel","visits","lifetime_spend","avg_ticket"]], drop_first=True)
y = df["responded"].astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print("train:", X_tr.shape, "  test:", X_te.shape)
print("train response rate:", y_tr.mean().round(3))
print("test  response rate:", y_te.mean().round(3))


## 4. Logistic regression

In [ ]:
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_tr_s, y_tr)
print(f"Train accuracy: {lr.score(X_tr_s, y_tr):.3f}")
print(f"Test  accuracy: {lr.score(X_te_s, y_te):.3f}")

print("\nCoefficients:")
for col, coef in zip(X.columns, lr.coef_[0]):
    print(f"  {col:35s}  {coef:+.3f}")


## 5. Decision tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_tr, y_tr)
print(f"Train accuracy: {dt.score(X_tr, y_tr):.3f}")
print(f"Test  accuracy: {dt.score(X_te, y_te):.3f}")

imp = pd.Series(dt.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index, imp.values, color="#0891B2")
ax.set_title("Decision-tree feature importances")
plt.tight_layout(); plt.show()


## Discussion prompts
1. Why is "accuracy" potentially misleading on this problem? (Hint: look at the response rate.)
2. Logistic regression and decision tree gave different views. Which would you trust to *explain* the result to Khun Ploy?
3. What features are missing that might lift performance?
